# Dipendenti pubblici 2021-2023 — preanalysis v0

**Domanda chiave:** il pubblico impiego sta tornando a crescere davvero, e in quali comparti si concentra la dinamica tra 2021 e 2023?

**Domande complementari:**
- quali comparti trainano di piu la crescita del personale?
- dove il saldo tra assunti e cessati migliora di piu?
- come cambia la composizione di genere tra i comparti principali?


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

PROJECT_DIR = Path.cwd().parent
REPO_ROOT = PROJECT_DIR.parents[2]
OUT_DIR = REPO_ROOT / 'out' / 'data' / 'mart' / 'dipendenti_pubblici_2021_2023'

frames = []
for year in [2021, 2022, 2023]:
    path = OUT_DIR / str(year) / 'mart_comparti_anno.parquet'
    df = pd.read_parquet(path)
    frames.append(df)

mart = pd.concat(frames, ignore_index=True)
mart.head()


In [ ]:
totali = (
    mart.groupby('anno', as_index=False)[['dipendenti_totali', 'assunti_totali', 'cessati_totali', 'saldo_netto']]
    .sum()
)
totali


In [ ]:
delta_comparti = (
    mart.pivot(index='comparto', columns='anno', values='dipendenti_totali')
    .fillna(0)
    .reset_index()
)
delta_comparti['delta_2023_vs_2021'] = delta_comparti[2023] - delta_comparti[2021]
delta_comparti.sort_values('delta_2023_vs_2021', ascending=False)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(totali['anno'], totali['dipendenti_totali'], marker='o', label='Dipendenti totali')
ax.bar(totali['anno'], totali['saldo_netto'], alpha=0.3, label='Saldo netto')
ax.set_title('Dipendenti pubblici: stock totale e saldo netto')
ax.set_xlabel('Anno')
ax.grid(axis='y', alpha=0.2, linestyle=':')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
top_2023 = mart[mart['anno'] == 2023].sort_values('dipendenti_totali', ascending=False).copy()
top_2023[['comparto', 'dipendenti_totali', 'assunti_totali', 'cessati_totali', 'saldo_netto', 'quota_donne_pct']]


In [ ]:
display(Markdown(
    f"""
### Prime evidenze

- i dipendenti totali passano da **{int(totali.loc[totali['anno'] == 2021, 'dipendenti_totali'].iloc[0]):,}** a **{int(totali.loc[totali['anno'] == 2023, 'dipendenti_totali'].iloc[0]):,}**
- il saldo netto totale passa da **{int(totali.loc[totali['anno'] == 2021, 'saldo_netto'].iloc[0]):,}** nel 2021 a **{int(totali.loc[totali['anno'] == 2023, 'saldo_netto'].iloc[0]):,}** nel 2023
- i comparti con la crescita assoluta piu forte tra 2021 e 2023 sono quelli in cima alla tabella `delta_comparti`
- nel 2023 `Istruzione e ricerca` e `Sanita'` restano i comparti piu grandi e con la dinamica piu leggibile
""".replace(',', '.')
))
